In [ ]:
# Exploration of the TBI cohort and ICU admission outcomes

In [ ]:
# Set up

# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# only 50 columns necessary printing DataFrames
# ensures wider MIMIC tables are not cut off
pd.set_option("display.max_columns", 50)

# Data paths

ED = "../data/mimic-iv-ed-2.2/ed/" # emergency department tables, including ED stays, triage, vital signs, diagnoses...
HOSP = "../data/mimic-iv-2.2/hosp/" # hospital tables, including admissions, patients, hospital diaognoses...
ICU = "../data/mimic-iv-2.2/icu/" # ICU tables reference ICU stays and is used to define outcome

# Settings for analysis:

# Define study cohort
# admitted_only = True -> keeps only TBI visits where patient is admitted to hospital
# admitted_only = False -> keeps all TBI visits, and patients discharged from ED are counted as a non-ICU outcome
admitted_only = True

# Define outcome window
# positive ICU outcome when patient is admitted to ICU within icu_windows_hours of arrival at ED
icu_windows_hours = 24

# Define vital signs derived from MIMIC-IV-ED triage and vitalsign tables
#   temperature: body temperature (Fahrenheit)
#   heartrate: heart rate (beats per minute)
#   resprate: respiratory rate (breaths per minute)
#   o2sat: oxygen saturation (%)
#   sbp: systolic blood pressure (mmHg) -> force exerted by blood against artery walls when heart beats
#   dbp: diastolic blood pressure (mmHg) -> force exerted by blood against artery walls when heart rests between beats
vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]

In [ ]:
# Load and inspect relevant tables
# Read from compressed CSV files and decompressed by pandas
# parse_dates converts from text into datetime objects for time difference (ED arrival to ICU admission) calculation

# ED module:
# edstays: one row per ED visit with arrival and departure times
edstays    = pd.read_csv(ED + "edstays.csv.gz", parse_dates=["intime", "outtime"])

# triage: vital signs and acuity recorded at ED arrival, pain is read as text (due to free text entries) and numbers
triage     = pd.read_csv(ED + "triage.csv.gz", dtype={"pain": str})

# vitalsign: repeated vital signs measurements during ED stay with time of recording
vitalsign  = pd.read_csv(ED + "vitalsign.csv.gz", parse_dates=["charttime"], dtype={"pain": str})

# diagnosis: ICD codes assigned during ED stay to identify TBI cases
diagnosis  = pd.read_csv(ED + "diagnosis.csv.gz", dtype={"icd_code": str})

# Hospital module:
# patients: patient demographics (sex, age, race) and death info
patients   = pd.read_csv(HOSP + "patients.csv.gz")

# admissions: admission and discharge times, used to link ED visits to inpatient stays
admissions = pd.read_csv(HOSP + "admissions.csv.gz", parse_dates=["admittime", "dischtime"])

# ICU module:
# icustays: one row per ICU stay with admission and discharge, used to define ICU admission outcome
icustays   = pd.read_csv(ICU + "icustays.csv.gz", parse_dates=["intime", "outtime"])

# dictionary to store tables for inspection
tables = {"edstays": edstays, "triage": triage, "vitalsign": vitalsign, "diagnosis": diagnosis,
          "patients": patients, "admissions": admissions, "icustays": icustays}

In [ ]:
def inspect(df, name):
    """
    Print table summary with size, column types, and percentage of missing values
    Print first five rows of the table for visual inspection
    """

    # rows = records, unique subject_id = distinct patients
    # if rows > patients, some patients appear more than once
    # :, formats numbers with commas for thousands
    print(f"{name}: {len(df):,} rows, {df['subject_id'].nunique():,} patients")
    
    # summary table with:
    #   dtype (data type),
    #   % missing (empty NaN values as percentage),
    #   and unique values (distinct values in column)
    # .T tranposes summary to read columns across page more easily
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "% missing": (df.isna().mean() * 100).round(1),
        "unique values": df.nunique(),
    }).T)
    display(df.head())

# Run for all loaded tables
for name, df in tables.items():
    inspect(df, name)

In [ ]:
# Data integrity checks

# confirm that tables are structured as assumed
# checks and prints True/False or percentage
# failure indicates merging will duplicate or lose patient records

# each ED visit (stay_id) should appear once in edstays
# is_unique returns True when if no stay_id is duplicated
print("edstays one row per stay:", edstays["stay_id"].is_unique)

# each ED visit (stay_id) should appear once in triage so merging does not duplicate visits
print("triage one row per stay:", triage["stay_id"].is_unique)

# every admitted ED visit (non-empty hadm_id) should link to hospital admission
# dropna() keeps only admitted visits, isin() checks if hadm_id exists in admissions, mean() gives the share that matched
# should be 100% if all admitted ED visits are linked to hospital admissions
# :.1% formats results as percentage to 1 decimal point
print("ED hadm_ids found in admissions:", f"{edstays['hadm_id'].dropna().isin(admissions['hadm_id']).mean():.1%}")

# Glasgow Coma Scale (GCS) is standard measure of TBI severity, but it is not recorded in MIMIC-IV-ED
# this searches column names case-insensitively for "gcs" and prints matches or "none"
# its absence is confirmed below so can be stated as a study limitation in report
print("GCS column recorded in triage:", [c for t in (triage, vitalsign) for c in t.columns if "gcs" in c.lower()] or "none")

In [ ]:
# TBI cohort building:

# ICD versions
# the database runs from 2011-2019. US hospitals switched from ICD-9 to ICD-10 coding in October 2015
# so TBI visits before the switch carry ICD 9 and after carry ICD 10
# the cohort requires both coding systems to cover the entire period

# remove stray spaces around code to match all texts
icd = diagnosis["icd_code"].str.strip()

# ICD 9 -> 850-854 are intracranial injuries (concussion, contusion, haemorrhage)
# str[:3] takes first three characters of code to include all subcodes eg 8500 will match 85401 will match
# results is True/False with one value per diagnosis row
tbi9 = (diagnosis["icd_version"] == 9) & icd.str[:3].isin(["850", "851", "852", "853", "854"])

# ICD 10 -> S06 is intracranial injury, the equivalent of ICD-9 850-854
# seventh character of S06 code gives encounter type
# encounter type A means the initial encounter
# limiting the cohort to initial encounter A (not follow-up visits) keeps first visit for injury
# excludes follow up visits (D) and late effects (S)
tbi10 = (diagnosis["icd_version"] == 10) & icd.str.startswith("S06") & icd.str.endswith("A")

# print number of distinct ED visits with at least one TBI code in each system
# actual TBI visit number, not number of codes
print("TBI diagnosis rows: ICD-9", tbi9.sum(), "and ICD-10", tbi10.sum())
print("ED stays with ICD-9 TBI code: ", diagnosis.loc[tbi9]["stay_id"].nunique())
print("ED stays with ICD-10 TBI code: ", diagnosis.loc[tbi10]["stay_id"].nunique())

In [ ]:
# attrition count of ED stays at each stage
# records how many ED visits and patients remain after each inclusion step
# forms the patient-flow attrition diagram in the report

attrition = [] # one entry per step and filled in by step() function

def step(df, label):
    attrition.append({"step": label, "ED stays": len(df), "patients": df["subject_id"].nunique()})
    
    # :<45 left aligns label in 45 character column
    # :>7 right aligns the count with thousand seperators so printed steps line up well
    print(f"{label:<45} {len(df):>7,} stays")

# start point for every ED visit
step(edstays, "All ED stays")

# keep only ED visits with at least one TBI code ie, ICD 9 or ICD 10 
# isin() checks each visit is counted once even with more than one TBI code
# .copy() makes the cohort an independent table and avoids pandas SettingWithCopy warnings for adding new columns
cohort = edstays[edstays["stay_id"].isin(diagnosis.loc[tbi9 | tbi10]["stay_id"])].copy()
step(cohort, "With a TBI diagnosis code")

# age is shifted by MIMIC to protect privacy
# calculated by anchor_age + (visit year - anchor_year) and filtered to age 18+
# ages above 89 are recorded as 91 in MIMIC, so oldest ages are approximate
# missing ages (NaN) result in False for >= 18 filter, so are removed
cohort = cohort.merge(patients[["subject_id", "anchor_age", "anchor_year"]], on="subject_id", how="left")
cohort["age"] = cohort["anchor_age"] + (cohort["intime"].dt.year - cohort["anchor_year"])
cohort = cohort[cohort["age"] >= 18].copy() # missing ages fail >= 18 filter
step(cohort, "Adults (18+) with known age")

# admitted_only = 
# True = keeps visits with hadm_id ie admitted to hospital from ED
# False = skips this so patients discharged from ED count as "no ICU"
if admitted_only:
    cohort = cohort[cohort["hadm_id"].notna()].copy()
    step(cohort, "Admitted to hospital")

# Double check merges have not caused duplication
assert cohort["stay_id"].is_unique

In [ ]:
# ICU admissions outcome:

# outcome -> whether patient is admitted to ICU within 24 hour window set by icu_windows_hours
# columns define label predicted but not used as model features to avoid leakage

# use first ICU stay of hospital admission
# groupby(hadm_id) groups by hospital admission
# .min() picks earliest ICU admission time
# rename column to icu_intime to stop clash with ED intime column
first_icu = (icustays.groupby("hadm_id", as_index = False)["intime"].min()
             .rename(columns = {"intime": "icu_intime"}))

# attach first ICU admission time to each cohort visit via hospital admission
# how = "left" keeps every visit and patients with no ICU admission have empty icu_intime
cohort = cohort.merge(first_icu, on = "hadm_id", how = "left")

# hours from ED arrival to first ICU admission
# time difference acquired by subtracting two datetimes and convert to seconds and divide by 3600 to get hours
# empty for no ICU admission patients
cohort["hours_to_icu"] = (cohort["icu_intime"] - cohort["intime"]).dt.total_seconds() / 3600

# icu_any: 1 if patient goes to ICU at any point, otherwise: 0
# notna() = True when ICU time exists and astype(int) converts True/False to 1/0
cohort["icu_any"] = cohort["icu_intime"].notna().astype(int)

# icu_24h: 1 when ICU admission is within 24 hour window, otherwise: 0
# later ICU admissions may reflect deterioration on ward, which cannot be predicted by triage data, so excluded
# between() includes both endpoints of range, and NaN return False so no ICU admission patients count as 0
cohort["icu_24h"] = cohort["hours_to_icu"].between(0, icu_windows_hours).astype(int)

# outcome summary with counts and percentage
print(f"ICU at any point: {cohort['icu_any'].sum():>5} ({cohort['icu_any'].mean():.1%})")
print(f"ICU within {icu_windows_hours} hours: {cohort['icu_24h'].sum():>5} ({cohort['icu_24h'].mean():.1%})")

# outcome for patients admitted after 24 hour window (ward deterioration)
print(f"ICU after {icu_windows_hours} hours: {(cohort['icu_any'] - cohort['icu_24h']).sum():>5}")

# distribution of time to ICU admission
# describe() ignores NaN values
# include 90th percentile to show tail length for late admissions
display(cohort["hours_to_icu"].describe(percentiles=[0.25, 0.5, 0.75, 0.90]).round(1).to_frame("hours from ED arrival to ICU"))


In [ ]:
# ICU timings: report figure:

# two figures to show when ICU admissions happen after ED arrival
# justifies 24 hour window for defining ICU outcome definition

h = cohort["hours_to_icu"].dropna() # dropna() keeps ICU patients only

# percentage of ICU admissions within window giving True/False values and calculated by mean()
pct_in_window = (h <= icu_windows_hours).mean() * 100 # percentage of ICU admissions within window

# side by side figures
fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 4))

# A: distribution over first 48 hours in one hour bins
# later admissions labelled in text to keep 24 hour window more readable
ax1.hist(h[h <= 48], bins = 48, range = (0, 48), color = "#3b6ea5")
ax1.axvline(icu_windows_hours, ls = "--", c = "k", lw = 1)
ax1.text(0.97, 0.95, f"{(h > 48).sum()} admissions after \n48 hours not shown",
         transform = ax1.transAxes, ha = "right", va = "top") # later admissions
ax1.set(xlabel = "Hours from ED arrival to ICU", ylabel = "ICU admissions",
        title = "A: Distribution of ICU admission times", xlim = (0, 48))

# B: cumulative percentage -> ICU admissions by hour
# values not capped at 72 hours - x-axis is limited so late admissions cannot create a false spike at the end
hs = np.sort(h)
ax2.plot(hs, np.arange(1, len(hs) + 1) / len(hs) * 100, c = "#3b6ea5", lw = 2)
ax2.axvline(icu_windows_hours, ls = "--", c = "k", lw = 1) # dash lines for readability
ax2.axhline(pct_in_window, ls = ":", c = "grey", lw = 1)
ax2.text(icu_windows_hours + 1.5, pct_in_window - 9,
         f"{pct_in_window:.1f}% within {icu_windows_hours} h", fontsize = 9)
ax2.set(xlabel = "Hours from ED arrival to ICU", ylabel = "Cumulative % of ICU admissions",
        title = "B: Cumulative ICU admissions", xlim = (0, 72), ylim = (0, 102)) # not capped at 100 as ylim goes above

# remove borders for readability
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout() # spacing
fig.savefig("../results/fig_icu_timing.png", dpi = 300, bbox_inches = "tight")   # high quality figure and trim whitespaces
plt.show()

In [ ]:
# Triage variables and missingness:

# add triage measurements to each ED visit in cohort
# triage table has one row per ED stay (double checked with integrity checks above)
# stay_id adds triage column without any duplication
# subject_id dropped because already included in cohort
# how = "left" keeps every visit even without triage records and gives NaN
cohort = cohort.merge(triage.drop(columns = "subject_id"), on = "stay_id", how = "left")

# flag rows with all six vitals missing
# isna() marks missing value as True and .all(axis = 1) gives True when all of row is True
# assumed that patients likely bypass normal triage due to urgency
cohort["all_vitals_missing"] = cohort[vitals].isna().all(axis = 1)
assert cohort["stay_id"].is_unique # duplication check

# percentage missing per variable -> overall and by outcome
# missing = True
# ICU and non-ICU columns would be similar if missingness was random and unrelated to outcome
# large differences -> missingness carries information about outcome?
# this affects how missing values are handled ie, as indicator variables rather than being filled in or dropped completely
miss = cohort[vitals + ["pain", "acuity", "chiefcomplaint"]].isna()
display(pd.DataFrame({
    "% missing": miss.mean() * 100,
    "% missing, no ICU": miss[cohort["icu_24h"] == 0].mean() * 100,
    "% missing, ICU": miss[cohort["icu_24h"] == 1].mean() * 100,
}).round(1))

In [ ]:
# Locating entirely blank triage rows:

# test whether blank triage indicates most critically ill patients
# eg taken to resuscitation or operation room

# number of stays and ICU rate in each group where all vitals missing = True
# higher ICU rate in True (all vitals missing) indicates blank triage is a signal of injury severity
g = cohort.groupby("all_vitals_missing")
display(g["icu_24h"].agg(stays="count", icu_rate="mean").round(3))

# acuity (Emergency Severity Index: 1 = most urgent, 5 = least) assigned by nurses during triage
# crosstab -> count visits for each combinatino of blank triage rows with acuity level columns
# normalise = "index" -> converts count to row percentages so each row sums to 1 instead of being dropped
# blank triage rows being concentrated in higher ESIs (1 and 2) supports explanation of triage bypass due to severity
# missing acuity is kept as its own column
print("Acuity (row percentage):")
display(pd.crosstab(cohort["all_vitals_missing"], cohort["acuity"].fillna("missing"), normalize = "index").round(2))

# arrival mode (eg ambulance, walk in, helicopter) shows what percentage of arrival type had blank triage row
# arrival mode is known at triage point so can be used as model feature
print("Arrival mode (row percentage):")
display(pd.crosstab(cohort["arrival_transport"], cohort["all_vitals_missing"], normalize = "index").round(2))

In [ ]:
# Explore later vitals testing:

# vitals might be charted in vitalsign table instead of initial triage form
# readings that are soon after arrival may be considered "at triage" and may fill otherwise empty gaps in triage vitals

# minutes from ED arrival to each vital sign reading
# reading matched to visit to get ED arrival time and difference converted to minutes
# keep only readings with matching ED stay via inner merge function
vs = vitalsign.merge(edstays[["stay_id", "intime"]], on = "stay_id")
vs["mins_after_arrival"] = (vs["charttime"] - vs["intime"]).dt.total_seconds() / 60

# readings timestamped before ED arrival were noted
# but because they are impossible, they are assumed errors
# count print but then remove
print("Readings before arrival (excluded):", (vs["mins_after_arrival"] < 0).sum())
vs_ok = vs[vs["mins_after_arrival"] >= 0]

# time of first valid reading for each stay (ie after ED arrival) found by .min()
# map() finds relevant stay_id
# no valid readings get NaN
cohort["first_vitals_mins"] = cohort["stay_id"].map(vs_ok.groupby("stay_id")["mins_after_arrival"].min())

# number of stays with no valid vitalsign readings split by blank triage group
# lambda s: s.isna().sum() counts missing NaN values  in first_vitals_mins within each group
print("\nNo vitalsign readings at all:")
display(cohort.groupby("all_vitals_missing")["first_vitals_mins"].apply(lambda s: s.isna().sum()))

# blank triage patients: how many were covered by readings after initial arrival but within certain time windows
# 15, 30, 60 minutes after ED arrival
# shows how many gaps can be filled and decide choice of cut off time
# when is too late for reading to still count as "at triage"?
blank = cohort["all_vitals_missing"]
for cut in (15, 30, 60):
    covered = blank & cohort["first_vitals_mins"].between(0, cut)
    print(f"Within {cut:>2} minutes of ED arrival: {covered.sum()} of {blank.sum()} blank-triage stays")

In [ ]:
# Exploration of data quality: implausible readings and free-text pain scores:

# 1st and 99th percentiles show realistic range for variables
# min and max include extreme values that may be recording errors which need to be removed before modelling

# describe() -> gives count, mean, sd, min, percentiles, max for columns
# transposed by .T for readability of rows
display(cohort[vitals + ["acuity"]].describe(percentiles=[.01, .5, .99]).T.round(1))

# pain recorded as free text, and is split into:
#     plain numbers -> eg 7
#     missing
#     other texts -> eg "unable" or "5/10"
# to_numeric -> "coerce" converts valid number entries and defines anything else as NaN
# therefore non-numerical and missing values become NaN
pain_num = pd.to_numeric(cohort["pain"], errors = "coerce")

# proportions
print("pain: numeric", f"{pain_num.notna().mean():.1%}", # converted successfully
      "| missing", f"{cohort['pain'].isna().mean():.1%}", # empty in original column
      "| other text", f"{(pain_num.isna() & cohort['pain'].notna()).mean():.1%}") # present in original column but did not convert successfully

# show 15 most common non-numerical entries
# .str.strip() removes stray spaces -> "unable" vs " unable"
# .str.lower() makes everything lowercase -> "unable" vs "Unable"
display(cohort.loc[pain_num.isna() & cohort["pain"].notna(), "pain"].str.strip().str.lower().value_counts().head(15))


In [ ]:
# Clean physiologically implausible values:

# ranges are purposefully generous to remove clear recording errors eg typos or wrong units
# keeps abnormal but possible values


# temperature is mostly Fahrenheit but some recorded in Celsius
# values between 30 and 45 are only plausible as Celsius, so are converted to Fahrenheit
# F = C x 9/5 + 32
celsius = cohort["temperature"].between(30, 45)
print("Temperature converted from Celsius:", celsius.sum())
cohort.loc[celsius, "temperature"] = cohort.loc[celsius, "temperature"] * 9 / 5 + 32

# plausibility limits for vital signs decided from clinical sources
# considers that patients may be in critical states during readings 
plausible = {
    "temperature": (90, 110),
    "heartrate":   (20, 250),
    "resprate":    (4, 60),
    "o2sat":       (50, 100),
    "sbp":         (40, 300),
    "dbp":         (20, 200),
}

# ~ -> "not" so ~between(low, high) is True for values outside plausibility limits
# notna() checks missing values are not counted as implausible
# print removed values in order
# replace with NaN to treat as missing rather than deleting entire visit from cohort
for col, (low, high) in plausible.items():
    bad = cohort[col].notna() & ~cohort[col].between(low, high)
    print(f"{col:<12} {bad.sum():>3} implausible values removed: {sorted(cohort.loc[bad, col].tolist())}")
    cohort.loc[bad, col] = np.nan

In [ ]:
# Assess pain score:

# many pain entries are non-numerical and most mean "unable to assess"
# keep as feature instead of being lost when column is converted to numericals

# text grouped as "unable to assess/report pain":
#   uta / ua / u/a / u / unable = unable to assess
#   ett / intubated = breathing tube in place
#   c / crit / critical = assumption of critical patient
# patients who cannot report pain usually have reduced consciousness -> signals more severe injury

# convert to text, remove stray spaces, make all lowercase
# astype(str) turns missing values to nan
pain_text = cohort["pain"].astype(str).str.strip().str.lower()
cohort["pain_unable"] = pain_text.str.match(r"^(uta|ua|u/a|u|unable|ett|intubated|crit|critical|c)$").astype(int)

# pain_score -> numeric pain score on 0-10 scale
# text becomes NaN and values outside 0-10 become errors and also set to NaN
cohort["pain_score"] = pd.to_numeric(cohort["pain"], errors="coerce")
cohort.loc[(cohort["pain_score"] < 0) | (cohort["pain_score"] > 10), "pain_score"] = np.nan

# number of stays and ICU rate for patients able vs unable to report on pain score
# higher ICU rate in unable group supports using pain_unable as a predictor of outcome
print(cohort.groupby("pain_unable")["icu_24h"].agg(stays="count", icu_rate="mean").round(3))

# triage_bypassed: 1 when all six vitals are missing, otherwise:0 
# likely that patient skipped triage
# True/False -> 1/0 makes it usable as model feature
cohort["triage_bypassed"] = cohort["all_vitals_missing"].astype(int)

# ICU rate for each combination of triage_bypassed and pain_unable
# shows whether two features overlap (same patients flagged by both) or carry separate info on outcome
print(cohort.groupby(["triage_bypassed", "pain_unable"])["icu_24h"].agg(stays = "count", icu_rate = "mean").round(3))

In [ ]:
# Compare ICU vs non-ICU patients:

# standardised mean different (SMD) measures how far apart two groups are by standard deviation (SD)
# unlike p values in that it functions indepedent of sample size
# |SMD| > 0.1 = meaningful difference, 0.5 = moderate difference, > 0.5 large difference

# vitals only comparable where recorded, ie, among most normally triaged patients (not blank triage patients)

# compare numeric variables -> age, six vital signs, pain score, acuity
numeric = ["age"] + vitals + ["pain_score", "acuity"]

# split into outcome groups
icu, no_icu = cohort[cohort["icu_24h"] == 1], cohort[cohort["icu_24h"] == 0]

def smd(a, b):
    """
    Standardised mean difference between two groups.
    Difference in means divided by average of two variances and square rooted.
    pandas ignores missing values in mean() and var() so only recorded values used.
    Positive SMD means ICU group has higher values.
    """
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

# summary table -> median in each group (less affected by extremes than mean) and SMD for each variable
# lower numbers are more urgent in acuity -> negative SMD means ICU patients were triaged as more severe
display(pd.DataFrame({
    "ICU median": icu[numeric].median(),
    "no ICU median": no_icu[numeric].median(),
    "SMD": [smd(icu[c], no_icu[c]) for c in numeric],
}).round(2))

# compare categorical variables -> number of stays and ICU rate in categories
# eg ICU rate for ambulance vs walk in arrivals
for col in ["gender", "arrival_transport"]:
    display(cohort.groupby(col)["icu_24h"].agg(stays="count", icu_rate="mean").round(3))

# binary flags already established -> percentage of each outcome group with flag
# percentages given by mean of 0/1 column as proportion of 1s
flags = ["triage_bypassed", "pain_unable"]
display(pd.DataFrame({
    "Percentage of ICU": icu[flags].mean() * 100,
    "Percentage of no ICU": no_icu[flags].mean() * 100
}).round(1))

In [ ]:
# Triage variables by outcome: report boxplots figure

# boxplot: median and interquartile ranges with whiskers showing typical range (1.5 times the interquartile range)
# outliers hidden by showfliers = False so boxes remain readable
# only patients with recorded values are included

# one small boxplot per variable compares ICU vs non-ICU patients

labels = {   # titles with units
    "age": "Age (years)", "temperature": "Temperature (°F)", "heartrate": "Heart rate (bpm)",
    "resprate": "Respiratory rate (/min)", "o2sat": "O₂ saturation (%)",
    "sbp": "Systolic BP (mmHg)", "dbp": "Diastolic BP (mmHg)",
    "pain_score": "Pain score (0–10)", "acuity": "ESI acuity (1 = most urgent)",
}

fig, axes = plt.subplots(3, 3, figsize = (11, 9)) # 3x3 grid for each numeric variable
for ax, col in zip(axes.flatten(), numeric): # axes.flatten() converts grid to list of 9 panels and zip pairs panel with variable
    # two boxes for ICU and no ICU, where dropna() removes missing values
    # patch_artist = True -> draws filled boxes to coloured in
    bp = ax.boxplot([icu[col].dropna(), no_icu[col].dropna()],
                    showfliers = False, patch_artist = True, widths = 0.6, medianprops = dict(color = "black"))

    for box, colour in zip(bp["boxes"], ["#c0392b", "#7f8c99"]):   # ICU red, no ICU grey
        box.set_facecolor(colour)
        box.set_alpha(0.6)
    ax.set_xticks([1, 2], ["ICU", "No ICU"])
    ax.set_title(labels[col], fontsize = 10)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout() # prevent overlapping
fig.savefig("../results/fig_triage_boxplots.png", dpi = 300, bbox_inches = "tight") # high quality figure
plt.show()



In [ ]:
# Explore base-line:

# how well does nurse-assigned acuity alone predict ICU admission?
# models to be built must clearly outperform this to add value beyond triage nurses' assessments

from sklearn.metrics import roc_auc_score

# AUROC cannot handle missing values so only use visits with recorded ESI data
has_esi = cohort["acuity"].notna()

# AUROC = area under ROC curve is probabiity of random ICU patient ranking as higher risk than random no ICU patient
# 0.5 = no better than chance, 1.0 = perfect rank

# minus sign FLIPS ESI scale because roc_auc_score expects higher scores to mean higher risk, but lower ESI means more urgent
# ESI has five levels so many patients share scores -> tied pairs counts as half correct
print("AUROC of ESI acuity alone:", round(roc_auc_score(cohort.loc[has_esi, "icu_24h"], -cohort.loc[has_esi, "acuity"]), 3))

# number of stays and ICU rate at each ESI level
# ICU rate should fall from ESI 1 to 5 if acuity operates functionally
display(cohort.groupby("acuity")["icu_24h"].agg(stays = "count", icu_rate = "mean").round(3))

# Full cohort baseline:

# first baseline drops stays with missing acuity which likely contain most critical patients who bypass triage
# fairer comparison against models trained on entire cohort -> missing acuity assumed to mean bypassed triage -> most urgent
# given ESI = 0 -> more urgent than 1
esi_filled = cohort["acuity"].fillna(0)
print("AUROC of ESI (missing = most urgent), full cohort:",
      round(roc_auc_score(cohort["icu_24h"], -esi_filled), 3))                                    

In [ ]:
# Summarise and save exploration:

# patient-flow table: number of ED stays and patients remaining after each cohort building step
# recorded by step()
# becomes flow diagram in report
# index = False -> stops pandas writing row numbers as extra columns in csv file
attrition_table = pd.DataFrame(attrition)
display(attrition_table)
attrition_table.to_csv("../results/attrition.csv", index = False)

# save cleaned cohort for preprocessing and modelling scripts
# save inside data/, which .gitignore excludes as it contains patient-level data!!
 
# makedirs -> creates interim folder
# exist_ok = True -> stops raising error if interim folder already exists
# pickle file keeps exact pandas data types which csv file would lose
import os
os.makedirs("../data/interim", exist_ok = True)
cohort.to_pickle("../data/interim/cohort.pkl")
print("Saved", len(cohort), "stays to data/interim/cohort.pkl")

# final check for repeated ED visits by same patient
# groupby("subject_id").size() -> counts number of stays per patient
# train/test sets must be split by patient (subject_id) not by stay -> so no patient appears in both
# if model recognised patient that it has already seen it would make test performance look better than it is 
visits = cohort.groupby("subject_id").size()
print("Patients:", len(visits), "| with more than one TBI visit:", (visits > 1).sum())

In [ ]:
######################
# Feature development:

# chief complaint terms
# choose which chief complaints are used as features based on ICU rates
# modelling decision so done using only training set

#
train = pd.read_pickle("../data/processed/train.pkl")
train_cohort = cohort[cohort["subject_id"].isin(train["groups"])]
print("Training stays used:", len(train_cohort)) # should print 1824

# Split complaints into their separate terms eg Fall, head injury becomes "fall", "head injury"
# Protects "/" so that the common term "s/p" (status post) is not split on the slash
text = train_cohort["chiefcomplaint"].str.lower().str.replace("s/p", "sp", regex=False)
words = text.str.split(r"[,/;]+").explode().str.strip()
cc = pd.DataFrame({"term": words, "icu": train_cohort.loc[words.index, "icu_24h"]})

# Terms that appear in > 20 stays in order of commonality and with ICU rate
display(cc.groupby("term")["icu"].agg(stays="count", icu_rate="mean")
        .query("stays >= 15").sort_values("stays", ascending=False).head(40).round(3))
